In [ ]:
from langchain_mistralai import MistralAIEmbeddings

In [ ]:
embed_model=MistralAIEmbeddings()

In [ ]:
def attach_embeddings(data: dict):
    claims = data.get("claims", [])
    texts = [c.get("canonical_text") for c in claims]
    embeddings = embed_model.embed_documents(texts)
    for idx, emb in enumerate(embeddings):
        claims[idx]["embedding"] = emb
    return data
embedding_chain=RunnableLambda(attach_embeddings)


In [ ]:
import numpy as np

def similarity_check(claim: dict, clusters: list, threshold: float = 0.75):

    claim_embedding = np.array(claim["embedding"])

    best_score = -1
    best_cluster_id = 0

    for cluster in clusters:
        cluster_embedding = np.array(cluster["embedding_parent"])

        score = np.dot(claim_embedding, cluster_embedding) / (
            np.linalg.norm(claim_embedding) * np.linalg.norm(cluster_embedding)
        )

        if score > best_score:
            best_score = score
            best_cluster_id = cluster["id"]

    if best_score < threshold:
        best_cluster_id = 0

    return {
        "score": round(float(best_score), 4),
        "cluster_id": best_cluster_id,
        "claim": claim
    }
similaritycheck_runnable = RunnableLambda(similarity_check)

In [ ]:
#how the clusters will be stored 

cluster = {
    "id": int,  
    "embedding_parent": list[float],  
    "participants": list[int]  
}